# Environments, Workspaces & CI/CD

## What's covered

- **The environment problem** — how the same configuration runs against dev, staging, and prod
- **Workspaces** — what they really are, and the common misunderstanding
- The **workspaces vs separate state** debate, and why production teams almost always pick separate state
- **Directory layouts** that scale — the `live/` + `modules/` pattern, the per-environment-directory pattern, the Terragrunt approach
- **CI/CD pipelines** for Terraform — automated `plan`, gated `apply`, drift checks
- **Secret handling** so credentials never end up in state or in pipeline logs
- The **tooling ecosystem** — Atlantis, Terragrunt, Spacelift, HCP Terraform, Env0


## The environment problem

Most systems run in at least three environments — development, staging, production. Each has slightly different inputs (smaller instance sizes in dev, more replicas in prod, different domain names, separate cloud accounts) but the same shape. How do you express that in Terraform?

Three classes of answer:

- **One state, branch the config with conditionals.** Same `.tf` files, the resources differ based on `var.env`. Single state file.
- **Workspaces.** One config, multiple state files. `terraform workspace new prod; terraform workspace select prod`. State is keyed by workspace.
- **Separate state files per environment.** One directory per environment (or one tfvars file per environment). Each environment has its own backend key.

The first option doesn't really work at any scale: dev and prod resources end up tangled in the same state, a `terraform destroy` against dev risks prod, and the conditionals get harder to read with every new variation. Skip it.

The choice is really between **workspaces** and **separate state**. The notebook walks both.


## Workspaces — what they really are

A workspace is a *named state file* in the same backend. The default workspace is called `default`. You create more with `terraform workspace new <name>`. Switching workspaces points the same configuration at a different state.

```bash
$ terraform workspace list
* default

$ terraform workspace new dev
$ terraform workspace new prod
$ terraform workspace select prod
$ terraform workspace show
prod
```

Inside the configuration, the **`terraform.workspace`** built-in returns the current workspace name. You can use it to vary inputs:

```hcl
locals {
  instance_count = terraform.workspace == "prod" ? 10 : 2
  instance_type  = terraform.workspace == "prod" ? "t3.large" : "t3.micro"
  domain         = "${terraform.workspace}.example.com"
}
```

**Where workspaces fit.** Genuinely parallel environments with *identical* shape and minor input differences. Per-developer sandbox states (`terraform workspace new alice`). Quick ephemeral environments for CI integration tests.

**Where workspaces hurt.** Production. Three structural problems make workspaces a bad fit for serious environments:

- **Same backend.** Production state, dev state, and Alice's sandbox state all live in the same S3 bucket under the same prefix. A misconfigured IAM policy that grants someone "dev" access actually grants them production state read access.
- **Same configuration.** A change to the prod resources is the same diff as a change to dev — you cannot test the prod config in isolation. The HCL is shared; only inputs differ.
- **Workspace switching is a runtime decision.** A teammate runs `terraform apply` after forgetting to `workspace select prod`. They're now applying against the wrong environment. There's no CI gate that can see the workspace mismatch from the file system alone.

The HashiCorp docs are explicit on this. The official guidance is: **workspaces are not the right solution for production environments**. Use separate state.


## Separate state — the production pattern

One directory (or set of directories) per environment, each with its own backend configuration pointing at a different state key. Same modules; different roots.

```
   live/
   ├── dev/
   │   ├── main.tf            <- module calls, ENV=dev
   │   ├── terraform.tfvars
   │   └── backend.tf         <- key = "live/dev/terraform.tfstate"
   ├── staging/
   │   ├── main.tf            <- same module calls, ENV=staging
   │   ├── terraform.tfvars
   │   └── backend.tf         <- key = "live/staging/terraform.tfstate"
   └── prod/
       ├── main.tf            <- same module calls, ENV=prod
       ├── terraform.tfvars
       └── backend.tf         <- key = "live/prod/terraform.tfstate"
   modules/
   ├── network/
   ├── compute/
   └── app/
```

Each environment is its own Terraform working directory, with its own state, its own backend, its own provider config (potentially its own *cloud account*). The shared shape lives in `modules/`, used by all three.

**The wins:**

- **Total isolation.** Destroying dev cannot touch prod. They share no state.
- **Different cloud accounts per environment.** Often the right answer for blast-radius reasons. Prod has its own AWS account; dev is in a sandbox account. The backend bucket itself can be in the prod account with cross-account read access for the others.
- **Different IAM policies per environment.** Engineers can have admin in dev, read-only in prod, and the same Terraform config respects that boundary at the CI layer.
- **CI can be explicit.** "This pipeline applies the `prod/` directory" is a more auditable statement than "this pipeline runs `terraform workspace select prod && terraform apply`."

**The cost:**

- **More boilerplate.** Each environment has a `main.tf` that mostly duplicates the others. The shared shape lives in modules, but the *module calls* are duplicated.
- **Coordinated upgrades.** Bumping a module version means updating it in dev, staging, prod separately. CI helps; tools like Terragrunt help more.

Despite the boilerplate, most production teams pick this pattern. The isolation is worth it.


## Directory layouts that scale

Beyond three environments, the layout becomes part of the problem. Three patterns are common.

### Pattern 1 — `live/` + `modules/`

The pattern above. Each environment is a directory under `live/`. Modules are factored into `modules/`. Per-environment differences live in `terraform.tfvars` or in the root `main.tf` of each environment.

**When this works:** small to medium teams, 3-5 environments. The `live/dev/main.tf` is a thin orchestration layer that mostly calls modules with environment-specific inputs.

### Pattern 2 — One directory per component, per environment

For larger orgs, "one Terraform root per environment" becomes too coarse — a change to the database shouldn't replan the entire network. Split further:

```
   live/
   ├── dev/
   │   ├── network/
   │   ├── compute/
   │   └── app/
   ├── staging/
   │   ├── network/
   │   ├── compute/
   │   └── app/
   └── prod/
       ├── network/
       ├── compute/
       └── app/
```

Each `network/`, `compute/`, `app/` is its own Terraform working directory, with its own state file. They reference each other via `terraform_remote_state` data sources.

```hcl
data "terraform_remote_state" "network" {
  backend = "s3"
  config = {
    bucket = "myorg-terraform-state"
    key    = "live/prod/network/terraform.tfstate"
    region = "us-east-1"
  }
}

resource "aws_instance" "web" {
  subnet_id = data.terraform_remote_state.network.outputs.public_subnet_ids[0]
  # ...
}
```

**When this works:** medium-to-large teams where different teams own different components, and changes to one component shouldn't replan the others.

### Pattern 3 — Terragrunt

[Terragrunt](https://terragrunt.gruntwork.io/) is a thin wrapper over Terraform that addresses the boilerplate problem of pattern 1/2. Each environment is a directory with a `terragrunt.hcl` that *references* a shared module config:

```hcl
# live/prod/network/terragrunt.hcl
include "root" {
  path = find_in_parent_folders()
}

terraform {
  source = "git::git@github.com:myorg/tf-modules.git//network?ref=v1.4.0"
}

inputs = {
  env  = "prod"
  cidr = "10.30.0.0/16"
  azs  = ["us-east-1a", "us-east-1b", "us-east-1c"]
}
```

Terragrunt generates the backend config, fetches the module, and runs Terraform. The `main.tf` boilerplate goes away.

**When Terragrunt fits:** large multi-environment, multi-component setups where the boilerplate of pure-Terraform pattern 2 would be hundreds of nearly-identical files. The cost is a dependency on Terragrunt itself; some teams find that boundary uncomfortable.


## CI/CD pipelines — the basic flow

A production-ready Terraform pipeline runs on every pull request, gates merges on a clean plan, and applies only on the `main` branch (or via a manual approval step).

```
   developer pushes PR
         |
         v
   +-----------------+
   | terraform fmt   |  fail if not formatted
   +-----------------+
         |
         v
   +-----------------+
   | terraform init  |  configure backend, fetch providers/modules
   +-----------------+
         |
         v
   +-----------------+
   | terraform       |  print plan in PR comment
   | plan -out=plan  |  fail if errors
   +-----------------+
         |
         | reviewer approves PR
         v
   +-----------------+
   | merge to main   |
   +-----------------+
         |
         v
   +-----------------+
   | terraform apply |  run against the saved plan from before merge,
   | plan            |  or re-plan and apply with manual approval
   +-----------------+
```

A few details worth getting right:

- **`terraform fmt -check`** catches unformatted code at the gate. No one's HCL is ever a debate again.
- **`terraform validate`** catches syntax errors and type mismatches without needing cloud auth. Cheap and fast.
- **`terraform plan -out=plan.tfplan`** in the PR pipeline, save the plan as an artifact. **`terraform apply plan.tfplan`** in the apply pipeline uses the *same* plan that was reviewed. No drift between what was approved and what runs.
- **`tflint` and `tfsec`** run static analysis on the HCL: missing `description` fields, deprecated provider syntax, security smells like open security groups, IAM wildcards. Run them on every PR.
- **`infracost`** estimates the dollar impact of the plan and surfaces it in the PR. Reviewers see "this change adds \$340/month" before approving.

**The apply step is the part that needs the most care.** Either an automatic apply against `main` (suitable for non-production environments) or a manual gate that a senior engineer approves (production). Whichever you pick, the gate should be auditable — who approved, what plan, when.


## Plan / apply gating in practice

Three common gating patterns:

### Plan in PR, apply on merge

The default for non-production:

```yaml
# .github/workflows/terraform.yml (sketch)
on:
  pull_request:    # plan only
  push:
    branches: [main]   # apply

jobs:
  plan:
    if: github.event_name == 'pull_request'
    steps:
      - uses: actions/checkout@v4
      - uses: hashicorp/setup-terraform@v3
      - run: terraform init
      - run: terraform plan -out=plan.tfplan
      - uses: actions/upload-artifact@v4
        with: { name: plan, path: plan.tfplan }

  apply:
    if: github.ref == 'refs/heads/main'
    needs: plan
    environment: dev    # GitHub environment gates can require approvals
    steps:
      - uses: actions/checkout@v4
      - uses: hashicorp/setup-terraform@v3
      - run: terraform init
      - run: terraform apply -auto-approve
```

### Plan in PR, comment on diff, manual apply

For production. The plan output is posted as a PR comment. After review, an operator triggers the apply job manually.

This is what tools like **Atlantis** automate: the bot reads the PR, runs `terraform plan`, posts the output as a comment. A reviewer types `atlantis apply` in the PR comments, and the bot executes against the merged code.

### Speculative plan vs run plan

HCP Terraform / Terraform Cloud distinguishes:

- **Speculative plan** — runs on PRs, shows the plan, never applies. Used for review.
- **Run plan** — runs on merge or trigger, can apply. The apply still requires manual approval for protected workspaces.

The cleanest model: every change goes through a speculative plan (no risk), gets approved, becomes a run plan (with explicit confirmation if production).


## Secret handling in pipelines

Pipelines need credentials to run Terraform. Two principles:

**1. Use short-lived, scoped credentials.** Long-lived static AWS access keys in CI are the largest single source of credential leaks. Modern pipelines use:

- **OIDC federation** — GitHub Actions, GitLab CI, CircleCI can assume an AWS IAM role via OIDC. No static credentials stored anywhere; the pipeline gets a short-lived session token.
- **HCP Terraform's Dynamic Credentials** — same idea, baked into HCP Terraform.
- **Vault** — central credential broker that the pipeline authenticates to (via OIDC, k8s service account, etc.) and gets time-bounded credentials.

```yaml
# GitHub Actions example with AWS OIDC
permissions:
  id-token: write
  contents: read

steps:
  - uses: aws-actions/configure-aws-credentials@v4
    with:
      role-to-assume: arn:aws:iam::123456789012:role/github-terraform
      aws-region: us-east-1
  - run: terraform apply -auto-approve
```

No `AWS_ACCESS_KEY_ID` ever stored as a secret. The OIDC assertion in the GitHub job token is what proves identity; the AWS role's trust policy validates the assertion and issues credentials.

**2. Application secrets pulled from a secrets manager at plan/apply time.** Don't pass database passwords through pipeline variables. Pull from AWS Secrets Manager, GCP Secret Manager, Vault, or HCP Terraform's encrypted variables. The pipeline grants the Terraform role permission to *read* the secret; the value never appears in a config file or a CI log.

**3. Mask sensitive values in logs.** Pipeline runners (Actions, GitLab, CircleCI) mask values registered as secrets in their logs. *Register them*. A plan output that prints `db_password = "literal-string"` because the variable wasn't marked sensitive is a credential leak in your build logs.


## The tooling ecosystem

A short tour of the production tooling that wraps Terraform:

### Atlantis

Open-source bot that listens to GitHub/GitLab webhooks, runs `terraform plan` on PRs, posts the output as a comment, and applies when an operator comments `atlantis apply`. Self-hosted. The simplest "PR-driven Terraform" tool. Great for small/medium teams.

### Terragrunt

A wrapper that addresses Terraform's lack of DRY for multi-environment setups. `terragrunt.hcl` files in per-environment directories reference shared module configs. Generates backend config, manages dependencies between root configs, supports parallel runs across environments. The de facto choice for large multi-environment deployments. Maintained by Gruntwork.

### Spacelift

Commercial. CI/CD purpose-built for IaC — Terraform, Pulumi, CloudFormation, Kubernetes manifests. Strong policy enforcement (OPA-based), drift detection built in, stack-of-stacks management. Used by larger orgs that have outgrown HCP Terraform's pricing or features.

### env0

Commercial. Similar space to Spacelift — managed runs, policy gates, cost estimation, drift detection. Often picked alongside HCP Terraform comparisons.

### HCP Terraform (formerly Terraform Cloud)

HashiCorp's own offering. Free tier for small teams; paid tiers add Sentinel policies, OPA, run tasks (third-party integrations), SSO. The default if your team is starting fresh and wants something managed.

### Scalr

Commercial. Strong on multi-tenant / hierarchical access (one Scalr account, many sub-orgs). Used by orgs with complex team structures.

### A picking heuristic

- **Just starting out, small team** — HCP Terraform free tier.
- **PR-driven, want self-hosted, small team** — Atlantis.
- **Multiple environments, many components, hate boilerplate** — Terragrunt (often combined with one of the others).
- **Large org, need policy enforcement, willing to pay** — Spacelift, env0, or HCP Terraform paid tier.

The honest framing: pick the lightest tool that solves your team's actual pain. Many teams successfully run plain Terraform + GitHub Actions for years; introducing a wrapper before you've felt the pain is usually wasted complexity.


## Forward

Notebook seven turns to **Advanced Features & Refactoring** — the features you reach for once the basics are comfortable. Data sources for reading resources Terraform doesn't manage. Dynamic blocks for programmatically generating nested configuration. Provisioners and the good reasons to avoid them. The `moved` and `import` blocks that make safe in-place refactoring possible (vs the old `terraform state mv` and `terraform import` commands). And the custom `precondition` / `postcondition` validation blocks that let you encode invariants directly in the configuration.
